# Release16 checkpoint distillation + YTD alignment contract v1

This notebook converts the latest failed full-actor/full-probe evidence into an explicit next-training contract. It does **not** retain an actor, promote a model, or use 2026 YTD replay labels as direct training data.

In [1]:

from __future__ import annotations

import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd

try:
    from IPython.display import Markdown, display
except Exception:
    Markdown = None
    display = None


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src' / 'soxl_quant').exists():
            return candidate
    raise RuntimeError(f'could not find repo root from {start}')


ROOT = find_repo_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.soxl_quant.app.state_objective_training_targets import (
    StateObjectiveCheckpointSelectionConfig,
    audit_state_objective_target_lift,
    audit_state_objective_target_scope,
    score_state_objective_checkpoint_candidates,
    summarize_state_objective_checkpoint_selection,
)

EXPERIMENT = 'release16_checkpoint_distillation_ytd_alignment_contract_v1'
CARD_DIR = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD_DIR.mkdir(parents=True, exist_ok=True)
CREATED_AT = datetime.now(timezone.utc).isoformat()

PATHS = {
    'source_target_contract': ROOT / 'runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_target_v1/drawdown_upside_bc_target_contract.csv',
    'source_target_conclusion': ROOT / 'runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_target_v1/conclusion.md',
    'checkpoint_metrics': ROOT / 'runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1/checkpoint_candidate_metrics.csv',
    'checkpoint_selection': ROOT / 'runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1/checkpoint_selection_by_fold.csv',
    'checkpoint_summary': ROOT / 'runtime/model_selection/card/release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1/summary.json',
    'post_fullprobe_summary': ROOT / 'runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1/summary.json',
    'post_fullprobe_gate_matrix': ROOT / 'runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1/candidate_gate_matrix.csv',
    'post_fullprobe_walkforward_metrics': ROOT / 'runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1/walkforward_candidate_metrics.csv',
    'ytd_position_comparison': ROOT / 'runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1/ytd_position_comparison_vs_soxlbacktest.csv',
    'ytd_underparticipation': ROOT / 'runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1/ytd_underparticipation_days.csv',
    'soxlbacktest_reference': ROOT / 'runtime/production/card/soxl_backtest_policy_reimplementation_v1/new_policy_daily.csv',
}
missing = {name: str(path.relative_to(ROOT)) for name, path in PATHS.items() if not path.exists()}
if missing:
    raise FileNotFoundError(missing)

pd.set_option('display.max_columns', 120)
print(f'ROOT={ROOT}')
print(f'CARD_DIR={CARD_DIR.relative_to(ROOT)}')


ROOT=/Users/rziiiii/dev/workspace/.worktrees/dashboard-v2-model-features-fix
CARD_DIR=runtime/model_selection/card/release16_checkpoint_distillation_ytd_alignment_contract_v1


## Experiment Governance

- **Canonical baseline:** `runtime/production/card/soxl_backtest_policy_reimplementation_v1/new_policy_daily.csv` converted SOXLBacktest policy reference.
- **Direct comparison baseline:** `runtime/model_selection/card/release16_state_head_post_fullprobe_objective_screen_v1` and checkpoint replay candidate metrics.
- **Role:** diagnose/contract, not promotion.
- **Target windows:** fold-local training targets through the existing release16 target contract; walk-forward replay windows `wf_2020`, `wf_2022`, `wf_2023_2024`; YTD 2026 only as a diagnostic recipe guard.
- **Accepted trade-off:** keep max drawdown under 20% while improving participation and relative return; do not consume OOS/YTD rows as direct labels.
- **Changed layer:** selection/training-objective contract.
- **External reference:** converted SOXLBacktest rule policy and existing qlib/release16 state-objective target screens.
- **Hypothesis:** the retained actor failed mostly from underparticipation and objective drift; checkpoint distillation plus explicit YTD participation recipe and drift stops should be required before any new actor retention.
- **Acceptance criteria for a future training run:** full fixed replay MaxDD < 20%, all walk-forward folds pass relative/return/drawdown gates, YTD return beats the SOXLBacktest qlib baseline and current-best floor, YTD position replay is available, and dashboard model replay source is runnable.
- **Failure mode:** a candidate can look acceptable in one fold or via checkpoint-only replay but still lack a single retained actor, YTD replay, or safe target scope.

In [2]:

targets = pd.read_csv(PATHS['source_target_contract'])
targets['datetime'] = pd.to_datetime(targets['datetime']).dt.normalize()
checkpoint_metrics = pd.read_csv(PATHS['checkpoint_metrics'])
checkpoint_selection = pd.read_csv(PATHS['checkpoint_selection'])
gate_matrix = pd.read_csv(PATHS['post_fullprobe_gate_matrix'])
walkforward_metrics = pd.read_csv(PATHS['post_fullprobe_walkforward_metrics'])
ytd_compare = pd.read_csv(PATHS['ytd_position_comparison'])
ytd_under = pd.read_csv(PATHS['ytd_underparticipation'])
soxl_ref = pd.read_csv(PATHS['soxlbacktest_reference'])
with PATHS['checkpoint_summary'].open() as f:
    checkpoint_summary = json.load(f)
with PATHS['post_fullprobe_summary'].open() as f:
    post_fullprobe_summary = json.load(f)

for frame in [checkpoint_metrics, checkpoint_selection, walkforward_metrics, ytd_compare, ytd_under, soxl_ref]:
    for col in ['datetime', 'date', 'start', 'end', 'daily_path_start', 'daily_path_end']:
        if col in frame.columns:
            frame[col] = pd.to_datetime(frame[col], errors='coerce')

print('targets', targets.shape, targets['datetime'].min().date(), targets['datetime'].max().date())
print('checkpoint_metrics', checkpoint_metrics.shape)
print('ytd_compare', ytd_compare.shape, ytd_compare['datetime'].min().date(), ytd_compare['datetime'].max().date())
print('soxl_ref', soxl_ref.shape, soxl_ref['date'].min().date(), soxl_ref['date'].max().date())


targets (3731, 109) 2010-03-11 2022-12-30
checkpoint_metrics (12, 50)
ytd_compare (109, 16) 2026-01-02 2026-06-09
soxl_ref (116, 13) 2026-01-02 2026-06-09


In [3]:

folds = []
fold_cols = ['fold', 'train_start', 'train_end', 'eval_start', 'eval_end']
for row in targets[fold_cols].drop_duplicates().sort_values(['fold', 'train_start', 'eval_start']).to_dict('records'):
    folds.append(
        {
            'name': str(row['fold']),
            'train_start': str(pd.Timestamp(row['train_start']).date()),
            'train_end': str(pd.Timestamp(row['train_end']).date()),
            'eval_start': str(pd.Timestamp(row['eval_start']).date()),
            'eval_end': str(pd.Timestamp(row['eval_end']).date()),
        }
    )

scope_audit = audit_state_objective_target_scope(targets, folds=folds)
lift_audit = audit_state_objective_target_lift(
    targets,
    target_position_priority='risk',
    baseline_risk_position_col='source_risk_target_position_before_drawdown_upside'
    if 'source_risk_target_position_before_drawdown_upside' in targets.columns
    else 'risk_target_position',
)

scope_audit.to_csv(CARD_DIR / 'target_scope_audit.csv', index=False)
lift_audit.to_csv(CARD_DIR / 'target_lift_audit.csv', index=False)

display(scope_audit.head(20)) if display else print(scope_audit.head(20).to_string(index=False))
display(lift_audit.head(20)) if display else print(lift_audit.head(20).to_string(index=False))


,window,target_scope,target_fold,fold,rows,train_overlap_rows,eval_overlap_rows,outside_train_eval_rows,leakage_gate,scope_issue,train_start,train_end,eval_start,eval_end
0,wf_2020_train,fold_local,wf_2020_train,wf_2020_train,2470,2470,0,0,True,,2010-03-11,2019-12-31,2020-01-01,2020-12-31
1,wf_2022_train,fold_local,wf_2022_train,wf_2022_train,505,505,0,0,True,,2020-01-02,2021-12-31,2022-01-03,2022-12-30
2,wf_2023_2024_train,fold_local,wf_2023_2024_train,wf_2023_2024_train,756,756,0,0,True,,2020-01-02,2022-12-30,2023-01-03,2024-12-31


,fold,window,rows,rebound_rows,downtrend_rows,rebound_downtrend_rows,current_positive_rebound_lift_rows,candidate_release_positive_lift_rows,mean_current_effective_lift,mean_candidate_release_lift,max_current_effective_lift,max_candidate_release_lift,lift_semantics_gate,release_candidate_gate,target_position_priority,min_lift
0,wf_2020_train,wf_2020_train,2470,42,873,30,9,9,0.085957,0.085957,0.568200,0.568200,True,True,risk,1.000000e-09
1,wf_2022_train,wf_2022_train,505,13,234,9,1,2,0.015671,0.031056,0.203723,0.203723,True,True,risk,1.000000e-09
2,wf_2023_2024_train,wf_2023_2024_train,756,165,478,10,152,152,0.181708,0.181708,0.503723,0.503723,True,True,risk,1.000000e-09


In [4]:

source_target_col = 'drawdown_upside_consumed_target_position' if 'drawdown_upside_consumed_target_position' in targets.columns else 'risk_target_position'
# Some legacy target rows do not carry model feature columns. Add fold-local price-derived
# diagnostics so the buy/sell contrast sidecar is still useful for dashboard review.
targets = targets.sort_values(['fold', 'datetime']).reset_index(drop=True)
price_by_fold = targets.groupby('fold')['price']
targets['derived_return_1d'] = price_by_fold.pct_change(1)
targets['derived_return_5d'] = price_by_fold.pct_change(5)
targets['derived_return_21d'] = price_by_fold.pct_change(21)
rolling_peak_63 = price_by_fold.transform(lambda s: s.rolling(63, min_periods=2).max())
targets['derived_drawdown_63'] = targets['price'] / rolling_peak_63 - 1.0
targets['derived_realized_vol_21'] = targets.groupby('fold')['derived_return_1d'].transform(lambda s: s.rolling(21, min_periods=5).std())
feature_cols = [
    col
    for col in [
        'mom_5', 'mom_21', 'drawdown_63', 'rv_21', 'vol_of_vol_21',
        'soxl_soxx_ratio_widen_21', 'soxl_soxx_ratio_z_252', 'soxx_mom_21',
        'strategy_drawdown', 'panic_score', 'forward_return_5', 'forward_return_10', 'forward_return_21',
        'derived_return_1d', 'derived_return_5d', 'derived_return_21d', 'derived_drawdown_63', 'derived_realized_vol_21',
    ]
    if col in targets.columns
]
contract_cols = [
    'datetime', 'candidate', 'fold', 'window', 'target_scope', 'train_start', 'train_end',
    'eval_start', 'eval_end', 'action', 'price', 'teacher_position', 'rebound_target_position',
    'risk_target_position', source_target_col, 'objective_weight', 'is_rebound_cluster',
    'is_downtrend_state', 'release_gate_state', 'drawdown_control_state', *feature_cols,
]
contract_cols = [col for col in contract_cols if col in targets.columns]
distillation_contract = targets[contract_cols].copy()
distillation_contract = distillation_contract.rename(columns={source_target_col: 'distillation_target_position'})
distillation_contract['label_role'] = 'fold_local_train_label'
distillation_contract['training_label_allowed'] = True
distillation_contract['contract_source'] = str(PATHS['source_target_contract'].relative_to(ROOT))
distillation_contract['target_semantics'] = 'release16_drawdown_upside_consumed_position'
distillation_contract['oos_or_ytd_direct_label_guard'] = 'allowed_train_dates_only'
distillation_contract = distillation_contract.sort_values(['fold', 'datetime']).reset_index(drop=True)
distillation_contract.to_csv(CARD_DIR / 'distillation_target_contract.csv', index=False)

print('distillation rows', len(distillation_contract))
display(distillation_contract.groupby('fold')['distillation_target_position'].describe()) if display else print(distillation_contract.groupby('fold')['distillation_target_position'].describe().to_string())


distillation rows 3731


,count,mean,std,min,25%,50%,75%,max
fold,,,,,,,,
wf_2020_train,2470.0,0.326960,0.192392,0.052000,0.242100,0.250000,0.495400,0.9155
wf_2022_train,505.0,0.198093,0.187444,0.047588,0.068887,0.146277,0.247364,1.0000
wf_2023_2024_train,756.0,0.253613,0.236883,0.033795,0.062500,0.125181,0.500000,1.0000


In [5]:

contrast_rows = []
if feature_cols:
    grouped = distillation_contract.groupby(['action', 'is_rebound_cluster', 'is_downtrend_state'], dropna=False)
    for key, group in grouped:
        action, rebound_state, downtrend_state = key
        row = {
            'dataset': 'distillation_target_contract',
            'action': str(action),
            'is_rebound_cluster': bool(rebound_state),
            'is_downtrend_state': bool(downtrend_state),
            'rows': int(len(group)),
            'mean_teacher_position': float(pd.to_numeric(group['teacher_position'], errors='coerce').mean()),
            'mean_distillation_target_position': float(pd.to_numeric(group['distillation_target_position'], errors='coerce').mean()),
            'mean_objective_weight': float(pd.to_numeric(group['objective_weight'], errors='coerce').mean()) if 'objective_weight' in group.columns else np.nan,
        }
        for col in feature_cols:
            row[f'mean_{col}'] = float(pd.to_numeric(group[col], errors='coerce').mean())
        contrast_rows.append(row)

buy_sell_feature_contrast = pd.DataFrame(contrast_rows).sort_values(['action', 'is_downtrend_state', 'is_rebound_cluster']).reset_index(drop=True)
buy_sell_feature_contrast.to_csv(CARD_DIR / 'buy_sell_feature_contrast.csv', index=False)
display(buy_sell_feature_contrast) if display else print(buy_sell_feature_contrast.to_string(index=False))


,dataset,action,is_rebound_cluster,is_downtrend_state,rows,mean_teacher_position,mean_distillation_target_position,mean_objective_weight,mean_mom_5,mean_mom_21,mean_drawdown_63,mean_rv_21,mean_vol_of_vol_21,mean_soxl_soxx_ratio_widen_21,mean_soxl_soxx_ratio_z_252,mean_soxx_mom_21,mean_strategy_drawdown,mean_panic_score,mean_forward_return_5,mean_forward_return_10,mean_forward_return_21,mean_derived_return_1d,mean_derived_return_5d,mean_derived_return_21d,mean_derived_drawdown_63,mean_derived_realized_vol_21
0,distillation_target_contract,BUY,True,False,7,0.186314,0.650000,8.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.042667,-120.317071,NaN,NaN,NaN,-0.129589,-0.265354,-0.192732,-0.330447,0.054070
1,distillation_target_contract,BUY,True,True,23,0.555126,0.627609,18.889318,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.210230,-110.946913,NaN,NaN,NaN,-0.081852,-0.176704,-0.211646,-0.336794,0.052299
2,distillation_target_contract,HOLD,False,False,1567,0.351643,0.351643,2.853861,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.037309,81.976928,NaN,NaN,NaN,0.005152,0.021920,0.082509,-0.097334,0.038313
3,distillation_target_contract,HOLD,True,False,5,0.577120,0.650000,8.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.043666,75.953238,NaN,NaN,NaN,-0.002165,0.041466,0.189583,-0.111268,0.051302
4,distillation_target_contract,HOLD,False,True,834,0.593095,0.260312,14.550483,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.232603,-0.740346,NaN,NaN,NaN,-0.000437,-0.003119,-0.022945,-0.246370,0.050832
5,distillation_target_contract,HOLD,True,True,7,0.629057,0.681314,19.796907,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.235263,24.290267,NaN,NaN,NaN,0.021299,0.035190,0.251731,-0.067182,0.044259
6,distillation_target_contract,SELL,False,False,15,0.629813,0.629813,5.666667,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.018452,107.533244,NaN,NaN,NaN,0.025283,0.095928,0.158121,-0.032582,0.041334
7,distillation_target_contract,SELL,False,True,9,0.627033,0.316667,10.593445,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.132362,105.433102,NaN,NaN,NaN,0.032979,0.087247,0.240667,-0.054139,0.040289
8,distillation_target_contract,TAKE_PROFIT,False,False,3,0.055233,0.055233,1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.038258,92.811038,NaN,NaN,NaN,-0.004976,-0.025978,0.204261,-0.065446,0.027432
9,distillation_target_contract,nan,False,False,390,0.326388,0.326388,155.245699,0.068598,0.199801,-0.033425,0.751276,0.169013,0.119994,1.216881,0.066786,NaN,NaN,-0.021888,-0.021257,-0.014068,0.017090,0.069670,0.212903,-0.033597,0.048302


In [6]:

ytd_alignment = ytd_compare.copy()
ytd_alignment['datetime'] = pd.to_datetime(ytd_alignment['datetime']).dt.normalize()
ytd_alignment['label_role'] = 'ytd_diagnostic_recipe_no_direct_training'
ytd_alignment['training_label_allowed'] = False
ytd_alignment['recipe_term'] = 'ytd_soxlbacktest_position_alignment_loss'
ytd_alignment['target_position'] = pd.to_numeric(ytd_alignment['ref_position'], errors='coerce').clip(0.0, 1.0)
ytd_alignment['observed_actor_position'] = pd.to_numeric(ytd_alignment['full_position'], errors='coerce').clip(0.0, 1.0)
ytd_alignment['absolute_position_gap'] = (ytd_alignment['target_position'] - ytd_alignment['observed_actor_position']).abs()
ytd_alignment['underparticipation_weight'] = np.where(ytd_alignment['underparticipation_state'].astype(bool), 3.0, 1.0)
ytd_alignment['recipe_objective_weight'] = ytd_alignment['underparticipation_weight'] * (1.0 + ytd_alignment['absolute_position_gap'])
ytd_alignment['oos_or_ytd_direct_label_guard'] = 'blocked_ytd_oos_recipe_only'
ytd_alignment['source'] = str(PATHS['ytd_position_comparison'].relative_to(ROOT))
ytd_cols = [
    'datetime', 'price', 'ref_position', 'full_position', 'target_position', 'observed_actor_position',
    'absolute_position_gap', 'underparticipation_state', 'large_gap_state', 'forward_price_return_5d',
    'forward_price_return_10d', 'recipe_objective_weight', 'recipe_term', 'training_label_allowed',
    'oos_or_ytd_direct_label_guard', 'source',
]
ytd_alignment[ytd_cols].to_csv(CARD_DIR / 'ytd_alignment_target_rows.csv', index=False)

ytd_alignment_summary = {
    'rows': int(len(ytd_alignment)),
    'start': str(ytd_alignment['datetime'].min().date()),
    'end': str(ytd_alignment['datetime'].max().date()),
    'mean_absolute_position_gap': float(ytd_alignment['absolute_position_gap'].mean()),
    'max_absolute_position_gap': float(ytd_alignment['absolute_position_gap'].max()),
    'underparticipation_rows': int(ytd_alignment['underparticipation_state'].astype(bool).sum()),
    'large_gap_rows': int(ytd_alignment['large_gap_state'].astype(bool).sum()),
    'underparticipation_forward_5d_mean': float(pd.to_numeric(ytd_under['forward_price_return_5d'], errors='coerce').mean()) if not ytd_under.empty else 0.0,
    'underparticipation_forward_10d_mean': float(pd.to_numeric(ytd_under['forward_price_return_10d'], errors='coerce').mean()) if not ytd_under.empty else 0.0,
    'training_label_allowed': False,
    'guard': 'YTD rows may only shape a future recipe/reward/checkpoint objective after split-aware implementation; do not feed them as direct labels.',
}
print(json.dumps(ytd_alignment_summary, indent=2))


{
  "rows": 109,
  "start": "2026-01-02",
  "end": "2026-06-09",
  "mean_absolute_position_gap": 0.24746334787644184,
  "max_absolute_position_gap": 0.5359,
  "underparticipation_rows": 20,
  "large_gap_rows": 40,
  "underparticipation_forward_5d_mean": 0.1906784568945215,
  "underparticipation_forward_10d_mean": 0.29001102012188984,
  "training_label_allowed": false,
  "guard": "YTD rows may only shape a future recipe/reward/checkpoint objective after split-aware implementation; do not feed them as direct labels."
}


In [7]:

selected_paths = checkpoint_selection[['fold', 'selected_candidate', 'selected_kind', 'selected_daily_path']].copy()
reference_rows = []
for row in selected_paths.to_dict('records'):
    path = ROOT / str(row['selected_daily_path'])
    if not path.exists():
        continue
    daily = pd.read_csv(path)
    daily['datetime'] = pd.to_datetime(daily['datetime']).dt.normalize()
    daily['fold'] = str(row['fold'])
    daily['candidate'] = str(row['selected_candidate'])
    daily['candidate_kind'] = str(row['selected_kind'])
    daily['source_path'] = str(path.relative_to(ROOT))
    daily['label_role'] = 'checkpoint_eval_replay_reference_no_direct_training'
    daily['training_label_allowed'] = False
    daily['oos_or_ytd_direct_label_guard'] = 'blocked_eval_replay_reference_only'
    reference_rows.append(daily)
checkpoint_reference_rows = pd.concat(reference_rows, ignore_index=True) if reference_rows else pd.DataFrame()
checkpoint_reference_rows.to_csv(CARD_DIR / 'checkpoint_distillation_reference_rows.csv', index=False)

print('checkpoint reference rows', checkpoint_reference_rows.shape)
display(checkpoint_reference_rows.groupby(['fold', 'candidate'])['position'].describe()) if display and not checkpoint_reference_rows.empty else print(checkpoint_reference_rows.head().to_string(index=False))


checkpoint reference rows (753, 15)


,,count,mean,std,min,25%,50%,75%,max
fold,candidate,,,,,,,,
wf_2020,bc_initialization,252.0,0.302925,0.198819,0.082773,0.097660,0.289365,0.445820,0.950000
wf_2022,bc_initialization,250.0,0.081384,0.061817,0.000000,0.054098,0.095129,0.095622,0.288578
wf_2023_2024,bc_initialization,251.0,0.216822,0.158947,0.000000,0.085216,0.224908,0.275000,0.663933


In [8]:

RELATIVE_TARGET_PP = 30.0
MAX_DD_CEILING_PCT = 20.0
RETURN_FLOOR_PCT = 0.0
CURRENT_BEST_FLOOR_PCT = float(post_fullprobe_summary['current_best_required_return_pct'])
SOXLBACKTEST_YTD_RETURN_PCT = float(post_fullprobe_summary.get('soxlbacktest_reference_ytd', {}).get('return_pct', 132.07797795135497))
FULL_ACTOR_YTD_RETURN_PCT = float(post_fullprobe_summary['full_actor_ytd']['return_pct'])

terms = checkpoint_metrics.copy()
terms['candidate'] = terms['candidate'].astype(str)
terms['fold'] = terms['fold'].astype(str)
terms['walkforward_relative_return_gap_penalty'] = (RELATIVE_TARGET_PP - pd.to_numeric(terms['test_relative_return'], errors='coerce')).clip(lower=0.0)
terms['walkforward_drawdown_gap_penalty'] = (pd.to_numeric(terms['test_max_dd'], errors='coerce') - MAX_DD_CEILING_PCT).clip(lower=0.0)
terms['walkforward_return_gap_penalty'] = (RETURN_FLOOR_PCT - pd.to_numeric(terms['test_return'], errors='coerce')).clip(lower=0.0)
terms['reference_relative_gap_penalty'] = (-pd.to_numeric(terms['model_vs_reference_relative_delta_pp'], errors='coerce')).clip(lower=0.0)
terms['official_daily_path_available'] = terms['daily_path'].map(lambda value: (ROOT / str(value)).exists())
terms['single_retained_actor_available'] = False
terms['ytd_replay_available'] = False
terms['ytd_soxlbacktest_position_alignment_loss'] = float(ytd_alignment['absolute_position_gap'].mean())
terms['ytd_underparticipation_rows'] = int(ytd_alignment_summary['underparticipation_rows'])
terms['ytd_return_gap_vs_soxlbacktest_pct'] = max(0.0, SOXLBACKTEST_YTD_RETURN_PCT - FULL_ACTOR_YTD_RETURN_PCT)
terms['ytd_return_gap_vs_current_best_floor_pct'] = max(0.0, CURRENT_BEST_FLOOR_PCT - FULL_ACTOR_YTD_RETURN_PCT)
terms['target_scope_guard'] = bool(scope_audit['leakage_gate'].all())
terms['target_lift_guard'] = bool(lift_audit['lift_semantics_gate'].all()) if not lift_audit.empty else False
terms['future_training_required'] = True
terms['promotion_allowed'] = False
terms['promotion_blocker'] = 'checkpoint_or_full_actor_missing_single_retained_actor_ytd_replay_and_all_wf_gates'
terms['official_daily_path_checkpoint_objective_loss'] = (
    terms['walkforward_relative_return_gap_penalty'] / RELATIVE_TARGET_PP
    + terms['walkforward_drawdown_gap_penalty'] / MAX_DD_CEILING_PCT
    + terms['walkforward_return_gap_penalty'] / 100.0
    + terms['reference_relative_gap_penalty'] / RELATIVE_TARGET_PP
    + terms['ytd_soxlbacktest_position_alignment_loss']
    + (~terms['official_daily_path_available']).astype(float) * 10.0
)
terms.to_csv(CARD_DIR / 'checkpoint_objective_terms.csv', index=False)

alignment_scores = terms.groupby('candidate', as_index=False).agg(
    fold_count=('fold', 'nunique'),
    alignment_loss=('official_daily_path_checkpoint_objective_loss', 'mean'),
    max_alignment_loss=('official_daily_path_checkpoint_objective_loss', 'max'),
    window_gate_loss=('candidate_gate', lambda s: int((~s.astype(bool)).sum())),
    official_daily_path_coverage=('official_daily_path_available', 'mean'),
    max_relative_gap_penalty=('walkforward_relative_return_gap_penalty', 'max'),
    max_drawdown_gap_penalty=('walkforward_drawdown_gap_penalty', 'max'),
    max_return_gap_penalty=('walkforward_return_gap_penalty', 'max'),
    ytd_alignment_loss=('ytd_soxlbacktest_position_alignment_loss', 'mean'),
)
alignment_scores['required_fold_count'] = 3
alignment_scores['coverage_gate'] = (alignment_scores['fold_count'] >= alignment_scores['required_fold_count']) & (alignment_scores['official_daily_path_coverage'] == 1.0)
alignment_scores['window_gate'] = alignment_scores['window_gate_loss'] == 0
alignment_scores['missing_target_ratio'] = 1.0
alignment_scores['rebound_position_mae'] = 0.0
alignment_scores['risk_cap_violation'] = alignment_scores['max_drawdown_gap_penalty'] / MAX_DD_CEILING_PCT
scored = score_state_objective_checkpoint_candidates(
    alignment_scores,
    config=StateObjectiveCheckpointSelectionConfig(
        alignment_loss_weight=1.0,
        window_gate_loss_weight=0.05,
        missing_target_penalty=3.0,
        rebound_mae_weight=1.0,
        risk_cap_violation_weight=2.0,
        legacy_score_reward=0.0,
    ),
)
checkpoint_selection_summary = summarize_state_objective_checkpoint_selection(scored)
scored.to_csv(CARD_DIR / 'checkpoint_objective_scores.csv', index=False)

display(terms[['fold', 'candidate', 'test_return', 'test_relative_return', 'test_max_dd', 'walkforward_relative_return_gap_penalty', 'walkforward_drawdown_gap_penalty', 'walkforward_return_gap_penalty', 'official_daily_path_checkpoint_objective_loss', 'candidate_gate']]) if display else print(terms.to_string(index=False))
display(scored) if display else print(scored.to_string(index=False))
print(json.dumps(checkpoint_selection_summary, indent=2))


,fold,candidate,test_return,test_relative_return,test_max_dd,walkforward_relative_return_gap_penalty,walkforward_drawdown_gap_penalty,walkforward_return_gap_penalty,official_daily_path_checkpoint_objective_loss,candidate_gate
0,wf_2020,bc_initialization,57.666518,-0.427053,19.326906,30.427053,0.000000,0.000000,1.261698,False
1,wf_2020,ppo_retained_iter_0_bc_equivalent,57.666518,-0.427053,19.326906,30.427053,0.000000,0.000000,1.261698,False
2,wf_2020,ppo_retained_iter_4,44.378996,-13.714574,25.585056,43.714574,5.585056,0.000000,1.983869,False
3,wf_2020,ppo_selected_best,44.378996,-13.714574,25.585056,43.714574,5.585056,0.000000,1.983869,False
4,wf_2022,bc_initialization,-14.873092,71.714980,15.617043,0.000000,0.000000,14.873092,0.396194,True
5,wf_2022,ppo_retained_iter_0_bc_equivalent,-14.873092,71.714980,15.617043,0.000000,0.000000,14.873092,0.396194,True
6,wf_2022,ppo_retained_iter_4,-18.398913,68.189159,19.877624,0.000000,0.000000,18.398913,0.509931,False
7,wf_2022,ppo_selected_best,-18.398913,68.189159,19.877624,0.000000,0.000000,18.398913,0.509931,False
8,wf_2023_2024,bc_initialization,2.635083,2.991718,13.624617,27.008282,0.000000,0.000000,1.284989,False
9,wf_2023_2024,ppo_retained_iter_0_bc_equivalent,2.635083,2.991718,13.624617,27.008282,0.000000,0.000000,1.284989,False


,candidate,fold_count,alignment_loss,max_alignment_loss,window_gate_loss,official_daily_path_coverage,max_relative_gap_penalty,max_drawdown_gap_penalty,max_return_gap_penalty,ytd_alignment_loss,required_fold_count,coverage_gate,window_gate,missing_target_ratio,rebound_position_mae,risk_cap_violation,score,checkpoint_objective_loss,checkpoint_selection_score
0,bc_initialization,3,0.980961,1.284989,2,1.0,30.427053,0.000000,14.873092,0.247463,3,True,False,1.0,0.0,0.000000,0.0,4.080961,-4.080961
1,ppo_retained_iter_0_bc_equivalent,3,0.980961,1.284989,2,1.0,30.427053,0.000000,14.873092,0.247463,3,True,False,1.0,0.0,0.000000,0.0,4.080961,-4.080961
2,ppo_retained_iter_4,3,1.388011,1.983869,3,1.0,43.714574,5.585056,18.398913,0.247463,3,True,False,1.0,0.0,0.279253,0.0,5.096517,-5.096517
3,ppo_selected_best,3,1.388011,1.983869,3,1.0,43.714574,5.585056,18.398913,0.247463,3,True,False,1.0,0.0,0.279253,0.0,5.096517,-5.096517


{
  "decision": "checkpoint_objective_no_promotion",
  "selected_candidate": "bc_initialization",
  "selected_checkpoint_objective_loss": 4.080960596658614,
  "selected_checkpoint_selection_score": -4.080960596658614,
  "selected_window_gate": false,
  "selected_coverage_gate": true,
  "selected_alignment_loss": 0.9809605966586142,
  "selected_window_gate_loss": 2.0,
  "selected_missing_target_ratio": 1.0,
  "next_mechanism": "train_state_objective_checkpoint_or_bc",
  "required_terms": [
    "checkpoint_objective_alignment_loss",
    "checkpoint_objective_walkforward_gate_loss",
    "checkpoint_objective_rebound_mae",
    "checkpoint_objective_risk_cap_violation"
  ]
}


In [9]:

drift_stop_rows = [
    {'rule_id': 'ppo_vs_bc_drawdown_drift_stop', 'metric': 'candidate_max_dd_pct_minus_bc_max_dd_pct', 'operator': '<=', 'limit': 0.0, 'reason': 'PPO must not increase max drawdown versus BC initialization in any fold before actor retention review.'},
    {'rule_id': 'ppo_vs_bc_relative_return_drift_stop', 'metric': 'candidate_relative_return_pp_minus_bc_relative_return_pp', 'operator': '>=', 'limit': 0.0, 'reason': 'PPO must not reduce relative return versus BC initialization when BC already fails a fold objective.'},
    {'rule_id': 'walkforward_max_drawdown_hard_stop', 'metric': 'test_max_dd_pct', 'operator': '<', 'limit': MAX_DD_CEILING_PCT, 'reason': 'User hard gate: drawdown under 20%.'},
    {'rule_id': 'walkforward_relative_return_hard_stop', 'metric': 'test_relative_return_pp', 'operator': '>=', 'limit': RELATIVE_TARGET_PP, 'reason': 'State objective must produce positive relative-return edge across all walk-forward folds.'},
    {'rule_id': 'ytd_replay_availability_stop', 'metric': 'single_retained_actor_ytd_replay_available', 'operator': '==', 'limit': True, 'reason': 'No checkpoint-only candidate can be retained without single actor replay through dashboard/model source.'},
]
drift_stop_rules = pd.DataFrame(drift_stop_rows)
drift_stop_rules.to_csv(CARD_DIR / 'drift_stop_rules.csv', index=False)

drift_stop_contract = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': 'ppo_drift_stop_contract_ready_no_actor_retention',
    'rules': drift_stop_rows,
    'next_run_must_stop_on_any_violation': True,
    'promotion_allowed': False,
    'actor_retention_allowed': False,
}
with (CARD_DIR / 'drift_stop_contract.json').open('w') as f:
    json.dump(drift_stop_contract, f, indent=2, sort_keys=True)

display(drift_stop_rules) if display else print(drift_stop_rules.to_string(index=False))


,rule_id,metric,operator,limit,reason
0,ppo_vs_bc_drawdown_drift_stop,candidate_max_dd_pct_minus_bc_max_dd_pct,<=,0.0,PPO must not increase max drawdown versus BC i...
1,ppo_vs_bc_relative_return_drift_stop,candidate_relative_return_pp_minus_bc_relative...,>=,0.0,PPO must not reduce relative return versus BC ...
2,walkforward_max_drawdown_hard_stop,test_max_dd_pct,<,20.0,User hard gate: drawdown under 20%.
3,walkforward_relative_return_hard_stop,test_relative_return_pp,>=,30.0,State objective must produce positive relative...
4,ytd_replay_availability_stop,single_retained_actor_ytd_replay_available,==,True,No checkpoint-only candidate can be retained w...


In [10]:

scope_training_allowed = bool(scope_audit['leakage_gate'].all())
lift_semantics_all_pass = bool(lift_audit['lift_semantics_gate'].all()) if not lift_audit.empty else False
all_checkpoint_windows_pass = bool(scored['window_gate'].all()) if not scored.empty else False
all_checkpoint_coverage_pass = bool(scored['coverage_gate'].all()) if not scored.empty else False
single_retained_actor_ytd_replay_available = False
promotion_allowed = bool(scope_training_allowed and all_checkpoint_windows_pass and all_checkpoint_coverage_pass and single_retained_actor_ytd_replay_available and FULL_ACTOR_YTD_RETURN_PCT >= SOXLBACKTEST_YTD_RETURN_PCT and FULL_ACTOR_YTD_RETURN_PCT >= CURRENT_BEST_FLOOR_PCT)
actor_retention_allowed = False

selection = {
    'experiment': EXPERIMENT,
    'created_at': CREATED_AT,
    'decision': 'release16_checkpoint_distillation_ytd_alignment_contract_no_actor_retention',
    'canonical_baseline': 'soxlbacktest_policy_reimplementation_v1',
    'direct_comparison_baseline': 'release16_state_head_post_fullprobe_objective_screen_v1',
    'role': 'diagnose_contract',
    'changed_layer': 'selection_training_objective_contract',
    'target_window': ['wf_2020', 'wf_2022', 'wf_2023_2024', 'ytd_2026_recipe_only'],
    'external_reference': 'converted SOXLBacktest qlib baseline and release16 state objective target screens',
    'closest_historical_success_case': 'release16_rulebase_reversal_drawdown_upside_checkpoint_metric_replay_v1 selected bc_initialization passed wf_2022 candidate gate with MaxDD 15.6170 and relative return 71.715 pp.',
    'closest_historical_failure_case': 'release16_state_head_post_fullprobe_objective_screen_v1 failed fixed MaxDD, YTD return alignment, and multiple WF gates; full actor YTD avg position was 0.0853 versus SOXLBacktest 0.1622.',
    'why_not_duplicate': 'This run does not rerun PPO or select a checkpoint. It converts the failure into an explicit trainable target contract, YTD recipe guard, checkpoint objective terms, and PPO drift-stop contract.',
    'hypothesis': 'Checkpoint distillation plus explicit YTD participation recipe and PPO drift stops are required before a retained actor can align with SOXLBacktest while preserving drawdown under 20%.',
    'acceptance_criteria_next_training': [
        'target_scope_audit leakage_gate passes for train labels',
        'all WF folds pass relative >= 30 pp, return >= 0, MaxDD < 20%',
        'single retained actor has YTD replay through dashboard/model source',
        'YTD return >= SOXLBacktest qlib baseline and current-best floor',
        'PPO does not worsen drawdown or relative return versus BC initialization',
    ],
    'expected_tradeoff': 'Increase participation in missed high-upside states without letting PPO drift above 20% drawdown.',
    'failure_mode': 'Checkpoint-only/fold-local evidence may still fail to become a single retained actor with YTD replay; YTD labels may leak if used directly.',
    'comparison_table_row_key': EXPERIMENT,
    'scope_training_allowed': scope_training_allowed,
    'lift_semantics_all_pass': lift_semantics_all_pass,
    'all_checkpoint_windows_pass': all_checkpoint_windows_pass,
    'all_checkpoint_coverage_pass': all_checkpoint_coverage_pass,
    'single_retained_actor_ytd_replay_available': single_retained_actor_ytd_replay_available,
    'promotion_allowed': promotion_allowed,
    'actor_retention_allowed': actor_retention_allowed,
    'promotion_blocker': 'no_single_retained_actor_ytd_replay_and_checkpoint_windows_do_not_all_pass',
    'selected_checkpoint_objective_candidate': checkpoint_selection_summary.get('selected_candidate', ''),
    'selected_checkpoint_objective_loss': checkpoint_selection_summary.get('selected_checkpoint_objective_loss'),
    'checkpoint_objective_summary': checkpoint_selection_summary,
}

summary = {
    **selection,
    'source_paths': {name: str(path.relative_to(ROOT)) for name, path in PATHS.items()},
    'output_paths': {
        'distillation_target_contract': str((CARD_DIR / 'distillation_target_contract.csv').relative_to(ROOT)),
        'target_scope_audit': str((CARD_DIR / 'target_scope_audit.csv').relative_to(ROOT)),
        'target_lift_audit': str((CARD_DIR / 'target_lift_audit.csv').relative_to(ROOT)),
        'checkpoint_distillation_reference_rows': str((CARD_DIR / 'checkpoint_distillation_reference_rows.csv').relative_to(ROOT)),
        'checkpoint_objective_terms': str((CARD_DIR / 'checkpoint_objective_terms.csv').relative_to(ROOT)),
        'checkpoint_objective_scores': str((CARD_DIR / 'checkpoint_objective_scores.csv').relative_to(ROOT)),
        'ytd_alignment_target_rows': str((CARD_DIR / 'ytd_alignment_target_rows.csv').relative_to(ROOT)),
        'buy_sell_feature_contrast': str((CARD_DIR / 'buy_sell_feature_contrast.csv').relative_to(ROOT)),
        'drift_stop_contract': str((CARD_DIR / 'drift_stop_contract.json').relative_to(ROOT)),
    },
    'distillation_contract_rows': int(len(distillation_contract)),
    'target_scope_audit': {
        'rows': int(len(scope_audit)),
        'blocked_groups': int((~scope_audit['leakage_gate'].astype(bool)).sum()),
        'eval_overlap_rows': int(pd.to_numeric(scope_audit['eval_overlap_rows'], errors='coerce').sum()),
    },
    'target_lift_audit': {
        'rows': int(len(lift_audit)),
        'groups_without_current_lift': int((~lift_audit['lift_semantics_gate'].astype(bool)).sum()) if not lift_audit.empty else 0,
        'groups_with_release_candidate_lift': int(lift_audit['release_candidate_gate'].astype(bool).sum()) if not lift_audit.empty else 0,
    },
    'ytd_alignment_summary': ytd_alignment_summary,
    'hard_gates': {
        'max_drawdown_pct': MAX_DD_CEILING_PCT,
        'walkforward_relative_return_pp': RELATIVE_TARGET_PP,
        'return_floor_pct': RETURN_FLOOR_PCT,
        'soxlbacktest_ytd_return_pct': SOXLBACKTEST_YTD_RETURN_PCT,
        'current_best_required_return_pct': CURRENT_BEST_FLOOR_PCT,
        'full_actor_ytd_return_pct': FULL_ACTOR_YTD_RETURN_PCT,
    },
    'next_mechanism': 'train_checkpoint_distilled_actor_with_ytd_participation_recipe_and_ppo_drift_stop',
    'required_next_terms': [
        'official_daily_path_checkpoint_objective_loss',
        'ytd_soxlbacktest_position_alignment_loss',
        'walkforward_relative_return_gap_penalty',
        'walkforward_drawdown_gap_penalty',
        'ppo_vs_bc_drawdown_drift_stop',
        'single_retained_actor_ytd_replay_gate',
    ],
    'conclusion': 'No actor retained. The next run should use the train-local distillation contract plus recipe-only YTD alignment and stop PPO drift before a full fixed/WF/YTD/dashboard replay review.',
}

def json_ready(value):
    if isinstance(value, dict):
        return {str(k): json_ready(v) for k, v in value.items()}
    if isinstance(value, list):
        return [json_ready(v) for v in value]
    if isinstance(value, tuple):
        return [json_ready(v) for v in value]
    if isinstance(value, (np.bool_, bool)):
        return bool(value)
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        if math.isnan(float(value)) or math.isinf(float(value)):
            return None
        return float(value)
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    return value

with (CARD_DIR / 'selection.json').open('w') as f:
    json.dump(json_ready(selection), f, indent=2, sort_keys=True)
with (CARD_DIR / 'summary.json').open('w') as f:
    json.dump(json_ready(summary), f, indent=2, sort_keys=True)

conclusion_md = f"""# {EXPERIMENT}

Decision: `{selection['decision']}`.

No actor is retained and promotion is blocked. The source train-label contract passes scope audit, but checkpoint candidates still do not pass all walk-forward windows and there is no single retained actor with YTD replay. YTD SOXLBacktest position rows are saved only as a recipe/guard, not direct labels.

Key evidence:
- Distillation target rows: {len(distillation_contract)}
- Scope audit blocked groups: {summary['target_scope_audit']['blocked_groups']}
- YTD mean absolute position gap: {ytd_alignment_summary['mean_absolute_position_gap']:.6f}
- YTD underparticipation rows: {ytd_alignment_summary['underparticipation_rows']}
- Selected checkpoint-objective diagnostic candidate: `{selection['selected_checkpoint_objective_candidate']}`
- Promotion allowed: `{promotion_allowed}`

Next action: train a single retained-actor candidate only after adding official daily path objective loss, recipe-only YTD participation alignment, and PPO-vs-BC drift stops; then rerun full fixed, walk-forward, YTD, and dashboard replay gates.
"""
(CARD_DIR / 'conclusion.md').write_text(conclusion_md, encoding='utf-8')

print(json.dumps(json_ready(summary), indent=2, sort_keys=True))


{
  "acceptance_criteria_next_training": [
    "target_scope_audit leakage_gate passes for train labels",
    "all WF folds pass relative >= 30 pp, return >= 0, MaxDD < 20%",
    "single retained actor has YTD replay through dashboard/model source",
    "YTD return >= SOXLBacktest qlib baseline and current-best floor",
    "PPO does not worsen drawdown or relative return versus BC initialization"
  ],
  "actor_retention_allowed": false,
  "all_checkpoint_coverage_pass": true,
  "all_checkpoint_windows_pass": false,
  "canonical_baseline": "soxlbacktest_policy_reimplementation_v1",
  "changed_layer": "selection_training_objective_contract",
  "checkpoint_objective_summary": {
    "decision": "checkpoint_objective_no_promotion",
    "next_mechanism": "train_state_objective_checkpoint_or_bc",
    "required_terms": [
      "checkpoint_objective_alignment_loss",
      "checkpoint_objective_walkforward_gate_loss",
      "checkpoint_objective_rebound_mae",
      "checkpoint_objective_risk_cap

In [11]:

final_conclusion = f"""
## Conclusion

Decision: `{summary['decision']}`. No actor is retained and promotion remains blocked.

This notebook produced a train-local `distillation_target_contract.csv`, guarded checkpoint objective terms, recipe-only YTD SOXLBacktest alignment rows, buy/sell feature contrast, and PPO drift-stop rules. The YTD rows are explicitly `training_label_allowed=False` and must not be used as direct labels. The next meaningful loop is a single retained-actor training run that consumes the train-local contract and stops on WF/YTD/dashboard replay violations.
"""
if display and Markdown:
    display(Markdown(final_conclusion))
else:
    print(final_conclusion)



## Conclusion

Decision: `release16_checkpoint_distillation_ytd_alignment_contract_no_actor_retention`. No actor is retained and promotion remains blocked.

This notebook produced a train-local `distillation_target_contract.csv`, guarded checkpoint objective terms, recipe-only YTD SOXLBacktest alignment rows, buy/sell feature contrast, and PPO drift-stop rules. The YTD rows are explicitly `training_label_allowed=False` and must not be used as direct labels. The next meaningful loop is a single retained-actor training run that consumes the train-local contract and stops on WF/YTD/dashboard replay violations.
